## Objective

- Build the DQN network architecture (2 hidden layers, ReLU)
- Implement forward pass, Huber loss, Adam optimizer
- Sanity-check the network can process a batch and backpropagate

## Results

- Network outputs Q-values of correct shape (batch_size, n_actions)
- Confirmed a single training step reduces loss on a dummy batch

## Conclusion

DQN architecture is verified and ready for replay buffer and target
network integration in upcoming commits.

# Week 3 — Deep Q-Network (DQN)

## 1. Imports & Network Sanity Check

In [ ]:
import sys
sys.path.append('../src')
import torch
import numpy as np
from dqn_agent import DQNAgent, DQNNetwork

agent = DQNAgent()
print(agent.policy_net)

## 2. Test Forward Pass on a Batch

In [ ]:
# Simulate a batch of 8 states
dummy_states = torch.tensor(
    np.random.uniform(low=[0, 0], high=[100, 30], size=(8, 2)),
    dtype=torch.float32
)

q_values = agent.policy_net(dummy_states)
print("Output shape:", q_values.shape)  # expect (8, 10)
print("Sample Q-values:\n", q_values[:2])

## 3. Test One Training Step (Loss Decreases)

In [ ]:
dummy_targets = torch.randn(8, 10)  # fake target Q-values for sanity check

loss_before = agent.loss_fn(agent.policy_net(dummy_states), dummy_targets).item()

agent.optimizer.zero_grad()
loss = agent.loss_fn(agent.policy_net(dummy_states), dummy_targets)
loss.backward()
agent.optimizer.step()

loss_after = agent.loss_fn(agent.policy_net(dummy_states), dummy_targets).item()

print(f"Loss before step: {loss_before:.4f}")
print(f"Loss after step:  {loss_after:.4f}")
print("Loss decreased:", loss_after < loss_before)

## 4. Load Best Q-Learning Agent as Comparison Baseline

In [ ]:
from pricing_env import PricingEnv
from q_learning_agent import QLearningAgent
from eval_utils import run_episodes

env = PricingEnv()

q_agent_loaded = QLearningAgent()
q_agent_loaded.load('../outputs/trained_qtable_best.npy')
q_agent_loaded.epsilon = 0.0  # greedy for evaluation

q_results = run_episodes(q_agent_loaded, env, n_episodes=100)
print(f"Q-Learning mean revenue (loaded): {q_results['mean_revenue']:.2f}")

## 5. Configurable Evaluation Harness for Any Agent

In [ ]:
def evaluate_agent(agent, env, n_episodes=500, agent_type="tabular"):
    """
    Configurable evaluation harness.
    agent_type: 'tabular' (uses .act(obs)) or 'dqn' (uses .act_greedy(obs))
    """
    act_fn = agent.act if agent_type == "tabular" else agent.act_greedy
    episode_rewards = []

    for _ in range(n_episodes):
        obs, info = env.reset()
        total_reward = 0
        done = False
        while not done:
            action = act_fn(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            done = terminated or truncated
        episode_rewards.append(total_reward)

    return {
        "episodes": n_episodes,
        "mean_revenue": float(np.mean(episode_rewards)),
        "std_revenue": float(np.std(episode_rewards)),
        "all_rewards": episode_rewards,
    }


# Test harness against both agent types
q_test = evaluate_agent(q_agent_loaded, env, n_episodes=50, agent_type="tabular")
dqn_test = evaluate_agent(agent, env, n_episodes=50, agent_type="dqn")

print("Q-Learning (via harness):", q_test["mean_revenue"])
print("Untrained DQN (via harness):", dqn_test["mean_revenue"])

## 6. Test Replay Buffer Integration

In [ ]:
from replay_buffer import ReplayBuffer

buffer = ReplayBuffer(capacity=5000)

# Fill buffer by running a few random episodes
obs, info = env.reset()
for _ in range(200):
    action = np.random.randint(0, 10)
    next_obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    buffer.push(obs, action, reward, next_obs, done)
    obs = next_obs if not done else env.reset()[0]

print("Buffer filled with", len(buffer), "transitions")

states, actions, rewards, next_states, dones = buffer.sample(batch_size=16)
print("Sample batch — states shape:", states.shape, "| actions shape:", actions.shape)

## 7. Q-Learning — 500 Evaluation Episodes & Sample Price Trajectories

In [ ]:
q_eval_results = run_episodes(q_agent_loaded, env, n_episodes=500)
print(f"Q-Learning mean revenue (500 episodes): {q_eval_results['mean_revenue']:.2f}")
print(f"Std dev: {q_eval_results['std_revenue']:.2f}")

## 8. Sample Price Trajectories (3 Episodes)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

for ep in range(3):
    obs, info = env.reset()
    prices = []
    done = False
    while not done:
        action = q_agent_loaded.act(obs)
        prices.append(action)
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
    plt.plot(prices, marker='o', label=f"Episode {ep+1}")

plt.title("Q-Learning Agent — Price Level Chosen Over Time (3 Sample Episodes)")
plt.xlabel("Step")
plt.ylabel("Price Level (0-9)")
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/qlearning_sample_trajectories.png', dpi=150)
plt.show()

## 9. Test Integrated DQN Training Loop (Buffer + Target Network)

In [ ]:
from dqn_agent import DQNAgent
from replay_buffer import ReplayBuffer

dqn_agent = DQNAgent()
train_buffer = ReplayBuffer(capacity=10000)

# Warm up buffer with random exploration
obs, info = env.reset()
for _ in range(500):
    action = dqn_agent.act(obs)
    next_obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    train_buffer.push(obs, action, reward, next_obs, done)
    obs = next_obs if not done else env.reset()[0]

print("Buffer size after warmup:", len(train_buffer))

losses = []
for step in range(200):
    loss = dqn_agent.train_step(train_buffer, batch_size=64)
    if loss is not None:
        losses.append(loss)

print(f"Trained {len(losses)} steps")
print(f"First loss: {losses[0]:.4f} | Last loss: {losses[-1]:.4f}")
print(f"Target network updated {dqn_agent.train_step_count // dqn_agent.target_update_freq} time(s)")